### Imports

In [1]:
import sys
import os

sys.path.append(os.path.abspath(".."))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt


from src.wti_utils import build_wti_ohlc_features, create_labels


### CSV-Daten laden

minütliche WTI Future Intraday Kurse in USD von 2011-2022

In [2]:
wti = pd.read_csv("C:/Users/02_Florian_Benutzer/Desktop/trump_oil_analysis_v2/00_data\wti.csv")

<>:1: SyntaxWarning: invalid escape sequence '\w'
<>:1: SyntaxWarning: invalid escape sequence '\w'
C:\Users\02_Florian_Benutzer\AppData\Local\Temp\ipykernel_23980\1190269471.py:1: SyntaxWarning: invalid escape sequence '\w'
  wti = pd.read_csv("C:/Users/02_Florian_Benutzer/Desktop/trump_oil_analysis_v2/00_data\wti.csv")


In [3]:
wti_v1 = wti.copy()

Zeitzone des Datensatzes bestimmen, da auf Kaggle nicht angegebem

In [4]:
# Einbruch der counts um 17 Uhr deutet auf NY Zeitzone hin

# Mit diesem Python-Code gruppierst du die Daten nach der Uhrzeit, die aktuell im Datensatz steht, und zählst, wie viele Einträge es pro Stunde gibt:

# Fall 1: Die Lücke ist bei 17 Uhr. -> Deine Daten sind bereits in der New York Zeitzone (EST/EDT).

wti_v1['time'] = pd.to_datetime(wti_v1['time'])

# Stunde extrahieren (0 bis 23)
wti_v1['hour'] = wti_v1['time'].dt.hour

# Anzahl der Datenpunkte pro Stunde zählen
print(wti_v1['hour'].value_counts().sort_index())

hour
0     145365
1     154919
2     171426
3     182765
4     183397
5     182139
6     181676
7     182371
8     184237
9     185107
10    185036
11    185006
12    184443
13    180690
14    176731
15    167613
16    147228
17     19011
18    134967
19    129162
20    158876
21    161296
22    153402
23    146162
Name: count, dtype: int64


In [5]:
# Nur Freitage filtern - da Börsenschluss im 17 Uhr - deutet auf NY Zeit hin

# schauen wir uns den letzten Datenpunkt vor dem Wochenende an. Freitags um 17:00 Uhr New York

freitage = wti_v1[wti_v1['time'].dt.dayofweek == 4]

# Die jeweils letzte Uhrzeit an jedem Freitag anzeigen
print(freitage.groupby(freitage['time'].dt.date)['time'].max().head(10))

time
2011-01-07   2011-01-07 17:00:00
2011-01-14   2011-01-14 16:59:00
2011-01-21   2011-01-21 17:00:00
2011-01-28   2011-01-28 16:59:00
2011-02-04   2011-02-04 16:59:00
2011-02-11   2011-02-11 16:59:00
2011-02-18   2011-02-18 16:59:00
2011-02-25   2011-02-25 16:59:00
2011-03-04   2011-03-04 16:59:00
2011-03-11   2011-03-11 16:59:00
Name: time, dtype: datetime64[us]


In [29]:
wti[6050:6100]

,time,open,high,low,close
6050,2011-01-07 16:58:00,88.39,88.40,88.39,88.40
6051,2011-01-07 16:59:00,88.39,88.39,88.38,88.38
6052,2011-01-07 17:00:00,88.39,88.39,88.39,88.39
6053,2011-01-09 20:15:00,89.65,89.65,89.63,89.63
6054,2011-01-09 20:16:00,89.62,89.62,89.59,89.60
6055,2011-01-09 20:17:00,89.59,89.61,89.58,89.58
6056,2011-01-09 20:18:00,89.60,89.61,89.59,89.61
6057,2011-01-09 20:19:00,89.61,89.61,89.61,89.61
6058,2011-01-09 20:20:00,89.61,89.61,89.60,89.61
6059,2011-01-09 20:21:00,89.60,89.61,89.60,89.60


In [15]:
wti_v1[wti_v1["time"].dt.hour == 8]

,time,open,high,low,close,hour
620,2011-01-03 08:00:00,92.180,92.180,92.170,92.180,8
621,2011-01-03 08:01:00,92.170,92.180,92.170,92.170,8
622,2011-01-03 08:03:00,92.180,92.190,92.180,92.180,8
623,2011-01-03 08:04:00,92.190,92.200,92.180,92.200,8
624,2011-01-03 08:05:00,92.220,92.270,92.200,92.270,8
...,...,...,...,...,...,...
3882542,2022-12-30 08:55:00,78.267,78.347,78.267,78.329,8
3882543,2022-12-30 08:56:00,78.328,78.342,78.302,78.337,8
3882544,2022-12-30 08:57:00,78.329,78.362,78.287,78.302,8
3882545,2022-12-30 08:58:00,78.292,78.327,78.274,78.309,8


Konvertierung in US/East und von da in UTC Timestamps

In [7]:
wti_v2 = wti.copy()

In [8]:
# 2. Den String in ein "naives" (zeitzonenloses) Datetime-Objekt umwandeln
# (Pandas erkennt das Format meistens automatisch)
wti_v2['time'] = pd.to_datetime(wti_v2['time'])

# 3. New York Zeit zuweisen (Lokalisieren)
# 'ambiguous='NaT'' fängt die doppelte Stunde bei der Umstellung im Herbst ab,
# 'nonexistent='NaT'' fängt die fehlende Stunde bei der Umstellung im Frühjahr ab.
wti_v2['timestamp_ny'] = wti_v2['time'].dt.tz_localize('America/New_York', ambiguous='NaT', nonexistent='NaT')

# 4. In UTC umrechnen
wti_v2['timestamp_utc'] = wti_v2['timestamp_ny'].dt.tz_convert('UTC')

# 5. (Optional) Wenn du das "+00:00" am Ende der UTC-Zeit nicht in der CSV haben willst,
# kannst du die Zeitzonen-Information wieder entfernen (die Zeit bleibt die von UTC):
# wti_v2['timestamp_utc_clean'] = wti_v2['timestamp_utc'].dt.tz_localize(None)

# Ergebnis überprüfen
#print(df[['Timestamp', 'Timestamp_UTC_clean']].head())

In [9]:
wti_v2.shape

(3883025, 7)

In [10]:
wti_v2 = (
    wti_v2
    .groupby('timestamp_utc', as_index=False)
    .agg({
        'open': 'first',
        'high': 'max',
        'low': 'min',
        'close': 'last'
    })
)

In [11]:
wti_v2.shape

(3882805, 5)

In [12]:
wti_v2

,timestamp_utc,open,high,low,close
0,2011-01-03 01:15:00+00:00,91.280,91.290,91.260,91.260
1,2011-01-03 01:16:00+00:00,91.260,91.260,91.250,91.260
2,2011-01-03 01:17:00+00:00,91.250,91.260,91.250,91.260
3,2011-01-03 01:18:00+00:00,91.270,91.270,91.260,91.260
4,2011-01-03 01:19:00+00:00,91.250,91.250,91.250,91.250
...,...,...,...,...,...
3882800,2022-12-30 21:53:00+00:00,80.407,80.407,80.367,80.392
3882801,2022-12-30 21:54:00+00:00,80.402,80.427,80.387,80.422
3882802,2022-12-30 21:55:00+00:00,80.417,80.447,80.402,80.407
3882803,2022-12-30 21:56:00+00:00,80.427,80.427,80.369,80.387


In [13]:
wti_v2.dtypes

timestamp_utc    datetime64[us, UTC]
open                         float64
high                         float64
low                          float64
close                        float64
dtype: object

In [15]:
wti_v2.isna().sum()

timestamp_utc    0
open             0
high             0
low              0
close            0
dtype: int64

In [16]:
wti_v2.to_csv(r"C:\Users\02_Florian_Benutzer\Desktop\trump_oil_analysis_v2\00_data\wti_v2.csv", sep=",", index=False, encoding="utf-8")

In [17]:
wti_v3 = wti_v2.copy()

# Feature Engineering

In [18]:
wti_v3 = build_wti_ohlc_features(wti_v3)

In [20]:
wti_v3 = create_labels(wti_v3)

In [22]:
wti_v3.columns

Index(['open', 'high', 'low', 'close', 'log_close', 'ret_1', 'hl_range',
       'oc_return', 'ret_5', 'ret_15', 'ret_30', 'ret_60', 'ret_240', 'mom_5',
       'mom_15', 'mom_60', 'vol_close_5', 'vol_parkinson_5', 'vol_close_15',
       'vol_parkinson_15', 'vol_close_30', 'vol_parkinson_30', 'vol_close_60',
       'vol_parkinson_60', 'vol_close_240', 'vol_parkinson_240', 'vol_gk',
       'vol_ratio_5_60', 'sma_5', 'ema_5', 'dist_ema_5', 'sma_15', 'ema_15',
       'dist_ema_15', 'sma_60', 'ema_60', 'dist_ema_60', 'sma_240', 'ema_240',
       'dist_ema_240', 'hl_momentum', 'body', 'upper_wick', 'lower_wick',
       'abs_ret', 'vol_cluster', 'vol_5', 'jump', 'hour', 'dayofweek',
       'hour_sin', 'hour_cos', 'dow_sin', 'dow_cos', 'us_session',
       'vol_regime', 'trend_60', 'trend_regime', 'y_up_1', 'y_down_1',
       'y_up_5', 'y_down_5', 'y_up_15', 'y_down_15', 'y_up_30', 'y_down_30',
       'y_up_60', 'y_down_60', 'y_up_240', 'y_down_240', 'y_up_720',
       'y_down_720', 'y_up_1440'

Gemini Ansatz für Triple Barrier Labeling

In [ ]:
import pandas as pd
import numpy as np

def apply_event_triple_barrier(df_prices, tweet_timestamps, x_minutes, pt_multiplier, sl_multiplier):
    """
    df_prices: DataFrame mit minütlichen WTI Preisen (Index: UTC Timestamp)
    tweet_timestamps: Liste oder Series von UTC Timestamps der Tweets
    x_minutes: Die vertikale Barriere (z.B. 60 für 1 Stunde)
    """
    labels = []
    
    for t in tweet_timestamps:
        # 1. Finde den exakten oder nächsten Preis-Zeitstempel zum Tweet
        if t not in df_prices.index:
            # Falls Markt geschlossen (Wochenende/Nacht), Event überspringen oder anpassen
            continue 
            
        p_t = df_prices.loc[t, 'close']
        
        # Volatilität (z.B. rollierende Standardabweichung) zur dynamischen Barrieren-Berechnung
        # Für ein einfaches Baseline-Modell tut es oft auch ein fixer Prozentsatz (z.B. 0.5%)
        volatility = df_prices.loc[t, 'daily_vol'] 
        
        upper_barrier = p_t * (1 + pt_multiplier * volatility)
        lower_barrier = p_t * (1 - sl_multiplier * volatility)
        vertical_barrier = t + pd.Timedelta(minutes=x_minutes)
        
        # Schaue in die Zukunft (Fenster von t bis t + x_minutes)
        future_prices = df_prices.loc[t:vertical_barrier, 'close']
        
        # Prüfe, welche Barriere zuerst getroffen wird
        first_touch_upper = future_prices[future_prices >= upper_barrier].index.min()
        first_touch_lower = future_prices[future_prices <= lower_barrier].index.min()
        
        # NaNs durch unendlich ersetzen für den Vergleich
        ft_upper = first_touch_upper if pd.notna(first_touch_upper) else pd.Timestamp.max
        ft_lower = first_touch_lower if pd.notna(first_touch_lower) else pd.Timestamp.max
        
        # Labeling Logik (Dein Ziel: 1 wenn es steigt, 0 wenn nicht)
        if ft_upper < ft_lower and ft_upper <= vertical_barrier:
            labels.append({'timestamp': t, 'label': 1}) # Profit-Target zuerst getroffen
        elif ft_lower < ft_upper and ft_lower <= vertical_barrier:
            labels.append({'timestamp': t, 'label': 0}) # Stop-Loss zuerst getroffen
        else:
            # Vertikale Barriere zuerst erreicht (Ablauf der Zeit x)
            # Hier entscheidest du: Ist der Preis nach x Minuten höher als am Anfang?
            final_price = future_prices.iloc[-1]
            label = 1 if final_price > p_t else 0
            labels.append({'timestamp': t, 'label': label})
            
    return pd.DataFrame(labels)